In [1]:
# Locate the repository when Jupyter starts in Notebooks/.
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'Notebooks': PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))

import h5py
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

fname = str(Path.home()/"packages/nexus/KingCRAB_ElectronTracks.h5")  

with h5py.File(fname, "r") as f:
    hits = pd.DataFrame(f["/MC/hits"][:])
    particles = pd.DataFrame(f["/MC/particles"][:])
    config = pd.DataFrame(f["/MC/configuration"][:])

# Decode byte-string columns
for df in [hits, particles, config]:
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].apply(
                lambda x: x.decode(errors="ignore") if isinstance(x, bytes) else x
            )

print("Datasets loaded:")
print("hits:", hits.shape)
print("particles:", particles.shape)
print("config:", config.shape)

Datasets loaded:
hits: (1, 9)
particles: (14149, 25)
config: (36, 2)


In [2]:
print(config.to_string(index=False))

                             param_key                            param_value
                            event_type                                  other
                            num_events                                      1
                          saved_events                                      1
          /PhysicsList/RegisterPhysics            G4EmStandardPhysics_option4
          /PhysicsList/RegisterPhysics                         G4DecayPhysics
          /PhysicsList/RegisterPhysics              G4RadioactiveDecayPhysics
          /PhysicsList/RegisterPhysics                           NexusPhysics
          /PhysicsList/RegisterPhysics                   G4StepLimiterPhysics
          /PhysicsList/RegisterPhysics                       G4OpticalPhysics
               /nexus/RegisterGeometry                               KingCRAB
              /nexus/RegisterGenerator                SingleParticleGenerator
     /nexus/RegisterPersistencyManager                     Persi

In [3]:
print("Particle counts:")
print(particles["particle_name"].value_counts())

print("\nCreator processes:")
print(particles["creator_proc"].value_counts())

print("\nFinal processes:")
print(particles["final_proc"].value_counts().head(20))

print("\nFinal volumes for optical photons:")
photons = particles[particles["particle_name"] == "opticalphoton"]
print(photons["final_volume"].value_counts().head(30))

print("\nEnergy deposited per event [MeV]:")
event_E = hits.groupby("event_id")["energy"].sum()
print(event_E)
print(event_E.describe())

Particle counts:
particle_name
opticalphoton    14132
ie-                 16
e-                   1
Name: count, dtype: int64

Creator processes:
creator_proc
Electroluminescence    14118
Clustering                16
Scintillation             14
none                       1
Name: count, dtype: int64

Final processes:
final_proc
Transportation    14131
Drift                16
NoProcess             1
OpAbsorption          1
Name: count, dtype: int64

Final volumes for optical photons:
final_volume
VESSEL               7001
ENDCAP_MINUS         2682
EL_MESH_ANODE        1917
EL_MESH_GATE         1845
FLANGE_RING_MINUS     551
ENDCAP_PLUS           127
FLANGE_RING_PLUS        8
FS_LENS                 1
Name: count, dtype: int64

Energy deposited per event [MeV]:
event_id
0    0.0004
Name: energy, dtype: float32
count    1.0000
mean     0.0004
std         NaN
min      0.0004
25%      0.0004
50%      0.0004
75%      0.0004
max      0.0004
Name: energy, dtype: float64


In [4]:
n_events = particles["event_id"].nunique()
n_photons = len(photons)

print("Number of events:", n_events)
print("Total optical photons:", n_photons)
print("Optical photons/event:", n_photons / n_events)

print("\nPhoton creator processes:")
print(photons["creator_proc"].value_counts())

Number of events: 1
Total optical photons: 14132
Optical photons/event: 14132.0

Photon creator processes:
creator_proc
Electroluminescence    14118
Scintillation             14
Name: count, dtype: int64


In [5]:
print("All photon final volumes:")
print(photons["final_volume"].value_counts().to_string())

# Try common possible detector/focal plane volume names
capture_names = ["FS_LENS", "Image-Intensifier"]

for name in capture_names:
    captured = photons[photons["final_volume"] == name]
    print(f"\n{name}:")
    print("captured:", len(captured))
    print("capture fraction:", len(captured) / len(photons) if len(photons) else 0)

All photon final volumes:
final_volume
VESSEL               7001
ENDCAP_MINUS         2682
EL_MESH_ANODE        1917
EL_MESH_GATE         1845
FLANGE_RING_MINUS     551
ENDCAP_PLUS           127
FLANGE_RING_PLUS        8
FS_LENS                 1

FS_LENS:
captured: 1
capture fraction: 7.076139258420605e-05

Image-Intensifier:
captured: 0
capture fraction: 0.0


In [6]:
detector_volume = "Image-Intensifier"  
captured = photons[photons["final_volume"] == detector_volume]

print("Captured photons:", len(captured))

if len(captured) > 0:
    plt.figure()
    plt.scatter(captured["final_x"], captured["final_y"], s=5)
    plt.xlabel("detector x [mm]")
    plt.ylabel("detector y [mm]")
    plt.axis("equal")
    plt.title(f"Captured photon positions on {detector_volume}")
    plt.show()
else:
    print("No photons captured in this detector volume.")

Captured photons: 0
No photons captured in this detector volume.
